# 07. 콘텐츠 기반 필터링 (CBF) — 최지훈 B4

제안서 3.2절. 상품 텍스트(`item_features`)를 TF-IDF로 바꾸고, 강의 module 6의 아이템 기반 CF 공식에
평점 유사도 대신 **콘텐츠 유사도**를 넣어 예측 평점을 만든다.

| 셀 | 내용 | 대략 (Video_Games) |
|---|---|---|
| 0~1 | 설정, 데이터 | 1분 |
| 2 | TF-IDF 행렬 생성·저장 | 1분 |
| 3 | 비슷한 상품 눈으로 점검 | 수 초 |
| 4 | 이웃 수 k 튜닝 (valid RMSE로 선택) | 1~3분 |
| 5 | 기준 모델·MF와 비교 (범위 × 구간) | 1분 |
| 6 | 예측값 저장: `cbf`, `mf_fb`(MF + 미등장 상품 CBF 대체) | 수 초 |
| 7 | Top-N용 유사도 원점수 정의 선택 (valid, 프로토콜 ③) | 셀 안에서 예상 시간 출력 |
| 8 | 설정·결과 저장 | 수 초 |

규칙 (docs/CONVENTIONS.md, docs/EVAL_RULES.md)
- 사용자 이력은 **train만** 쓴다 (MF와 같은 조건)
- k와 원점수 정의는 **valid로만** 고른다. test는 비교·구성요소 분석 보고용
- 예측값 파일은 `metrics.save_predictions` 형식 (`user_id, item_id, uidx, iidx, pred`)

In [ ]:
# ===== 0. 설정 =====
import os, sys, subprocess, json, time, importlib
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/team10-recsys'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/qhfl3644/team10-recsys.git', REPO], check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
except ImportError:
    REPO = os.environ.get('TEAM10_REPO', '..')
sys.path.insert(0, REPO)
from config import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import src.models.cbf as cbf_mod
importlib.reload(cbf_mod)                  # 레포를 pull한 뒤 다시 실행해도 새 코드가 쓰이도록
from src.eval import metrics as M
from src.models.baselines import BASELINES
from src.models.cbf import CBF, build_tfidf, save_tfidf, similar_items, TFIDF_DEFAULT

DS = 'Video_Games'                         # 구축용. Toys 일반화(11/5)는 여기만 바꿔서 다시 실행
TFIDF = dict(TFIDF_DEFAULT)                # stop_words='english', min_df=3, max_df=0.5
KS = [1, 2, 3, 5, 10, None]                # 평점 예측 이웃 수 후보 (None = 평가한 상품 전부)
SCORE_KS = [1, 3, 5, None]                 # Top-N 원점수: 상위 몇 개 유사도의 평균 (1 = 최댓값)
KEYWORDS = {'Video_Games': ['mario kart', 'zelda', 'call of duty', 'minecraft', 'fifa'],
            'Toys_and_Games': ['lego', 'barbie', 'puzzle', 'nerf', 'hot wheels']}
OUT = f'{RESULT_DIR}/cbf'
os.makedirs(OUT, exist_ok=True)
klabel = lambda k: 'all' if k is None else str(k)
print('준비 완료:', DS)

## 1. 데이터

`item_features.parquet`의 행 순서가 `item_map`의 iidx와 같아야 TF-IDF 행 번호 = 상품 번호가 된다.

In [ ]:
# ===== 1. 데이터 =====
t0 = time.time()
data = M.load_data(DS)
feat = pd.read_parquet(proc_path(DS, 'item_features'))
assert (feat['iidx'].values == np.arange(data.n_items)).all(), 'item_features가 iidx 순서가 아닙니다'
assert (feat['item_id'].astype(str).values == data.item_map['item_id'].astype(str).values).all(), 'item_map과 상품 순서가 다릅니다'
print(f'{DS}: 사용자 {data.n_users:,} / 상품 {data.n_items:,}')
print(f'  train {len(data.train):,} / valid {len(data.valid):,} / test {len(data.test):,}')
print(f'  train에 없는 상품이 정답인 행: valid {int((~data.seen_mask(data.valid)).sum())} / test {int((~data.seen_mask(data.test)).sum())}')
print(f'  ({time.time()-t0:.0f}초)')

## 2. TF-IDF 행렬

- `stop_words='english'`: the, and 같은 불용어 제거
- `min_df=3`: 상품 3개 미만에만 나오는 단어 제거 (오타·모델 번호 같은 잡음)
- `max_df=0.5`: 상품 절반 넘게 나오는 단어 제거 (game 같은 너무 흔한 단어)
- L2 정규화로 행 길이를 1로 맞춰서 **내적 = 코사인 유사도**

상품 텍스트는 평점과 무관한 상품 정보라 전체 상품으로 TF-IDF를 만든다 (valid·test 평점은 쓰지 않음).

In [ ]:
# ===== 2. TF-IDF =====
t0 = time.time()
X, vec = build_tfidf(feat['item_features'].values, **TFIDF)
nnz_row = np.diff(X.indptr)
print(f'TF-IDF: {X.shape[0]:,} 상품 × {X.shape[1]:,} 단어, 0이 아닌 값 {X.nnz:,}개')
print(f'  상품당 단어 수: 평균 {nnz_row.mean():.0f}, 중앙값 {np.median(nnz_row):.0f}')
print(f'  단어가 하나도 안 남은 상품: {int((nnz_row == 0).sum())}개 (이 상품은 유사도가 모두 0 → 사용자 평균으로 예측)')
idf = vec.idf_
terms = vec.get_feature_names_out()
print('  가장 흔한 단어 (IDF 낮음):', ', '.join(terms[np.argsort(idf)[:15]]))
TFIDF_PATH = f'{PROC_DIR}/{DS}/tfidf.npz'
save_tfidf(TFIDF_PATH, X, vec, TFIDF)
print(f'  저장: {TFIDF_PATH}  ({time.time()-t0:.0f}초)')

## 3. 비슷한 상품 눈으로 점검

인기 상품 3개와 키워드로 찾은 상품의 이웃 5개를 본다. 같은 시리즈·장르·플랫폼이 나오면 정상이다.

In [ ]:
# ===== 3. 눈으로 점검 =====
titles = feat['title'].fillna('').astype(str).values
low = pd.Series(titles).str.lower()

def show(i, n=5):
    nb, s = similar_items(X, int(i), n)
    print(f'■ {titles[i][:80]}  (train 평점 {data.item_count[i]}개)')
    for j, v in zip(nb, s):
        print(f'    {v:.3f}  {titles[j][:80]}')

for i in np.argsort(-data.item_count)[:3]:
    show(i)
for kw in KEYWORDS.get(DS, []):
    hit = np.flatnonzero(low.str.contains(kw, regex=False).values)
    if len(hit):
        show(hit[np.argmax(data.item_count[hit])])
    else:
        print(f'(키워드 "{kw}" 상품 없음)')

## 4. 이웃 수 k 튜닝

사용자가 평가한 상품 중 대상과 가장 비슷한 k개로 가중평균한다. **valid RMSE가 가장 낮은 k를 고른다.**
test 값은 구성요소 분석(제안서 5.2절, k 변화 그래프)용으로 같이 계산하지만 선택에는 쓰지 않는다.

In [ ]:
# ===== 4. k 튜닝 =====
t0 = time.time()
cbf = CBF(X, k=5, score_k=5).fit(data.train, data.n_users, data.n_items)
uv, iv, yv = data.valid['uidx'].values, data.valid['iidx'].values, data.valid['rating'].values
ut, it, yt = data.test['uidx'].values, data.test['iidx'].values, data.test['rating'].values
pv = cbf.predict_ks(uv, iv, KS)
pt = cbf.predict_ks(ut, it, KS)
k_table = pd.DataFrame([dict(k=klabel(k), valid_rmse=M.rmse(yv, pv[k]), valid_mae=M.mae(yv, pv[k]),
                             test_rmse=M.rmse(yt, pt[k]), test_mae=M.mae(yt, pt[k])) for k in KS])
BEST_K = KS[int(k_table['valid_rmse'].values.argmin())]
cbf.k = BEST_K
print(k_table.round(4).to_string(index=False))
print(f'\n선택: k = {klabel(BEST_K)}  (valid RMSE {k_table["valid_rmse"].min():.4f})')
print(f'사용자 평균으로 대체된 비율 (유사도 합 0): valid {cbf.fallback_rate(uv, iv)*100:.2f}%, test {cbf.fallback_rate(ut, it)*100:.2f}%')
print(f'({time.time()-t0:.0f}초)')

fig, ax = plt.subplots(figsize=(6, 3.6))
xs = np.arange(len(KS))
ax.plot(xs, k_table['valid_rmse'], marker='o', linewidth=2, color='#2a78d6', label='valid')
ax.plot(xs, k_table['test_rmse'], marker='o', linewidth=2, color='#eb6834', label='test')
ax.set_xticks(xs, [klabel(k) for k in KS])
ax.set_xlabel('Neighbors k (most similar rated items)')
ax.set_ylabel('RMSE')
ax.set_title(f'CBF: RMSE by number of neighbors ({DS})')
ax.spines[['top', 'right']].set_visible(False); ax.grid(alpha=0.3); ax.legend(frameon=False)
fig.savefig(f'{OUT}/{DS}_cbf_k.png', dpi=200, bbox_inches='tight'); plt.show()

## 5. 기준 모델·MF와 비교 (test)

- `all`: test 전체 / `seen`: 정답 상품이 train에 있는 행만 (순수 MF 비교용)
- 구간: train 평점 수 cold(3~4) / mid(5~9) / active(10+)
- MF 예측값은 혜림 님이 `predictions/`에 올린 `mf_test`를 그대로 읽는다

In [ ]:
# ===== 5. 비교 =====
base = {k: c().fit(data.train, data.n_users, data.n_items) for k, c in BASELINES.items() if k != 'popularity'}
p_base = {k: m.predict(ut, it) for k, m in base.items()}
p_mf = M.load_predictions(DS, 'mf', 'test', data.test)
rt = M.rating_table({**p_base, 'mf': p_mf, 'cbf': pt[BEST_K]}, data.test, data)
print('RMSE (test)')
print(rt.pivot_table(index=['scope', 'group'], columns='model', values='rmse', sort=False)
        [['global_mean', 'user_mean', 'item_mean', 'mf', 'cbf']].round(4).to_string())

un = ~data.seen_mask(data.test)
print(f'\ntrain에 없는 상품이 정답인 test {int(un.sum())}행: MF RMSE {M.rmse(yt[un], p_mf[un]):.4f}  vs  CBF RMSE {M.rmse(yt[un], pt[BEST_K][un]):.4f}')

crit = M.criterion_rating_vs_baselines(pt[BEST_K], p_base, data.test, data)
print(f'\n(참고) CBF vs 기준 모델 3개: {"모두 낮음" if crit.attrs["passed"] else "모두 낮지는 않음"}')
print(crit[['baseline', 'rmse_model', 'rmse_baseline', 'ci_lo', 'ci_hi', 'passed']].round(4).to_string(index=False))

## 6. 예측값 저장

- `cbf_valid`, `cbf_test`: CBF 단독 예측
- `mf_fb_valid`, `mf_fb_test`: MF 예측에서 **train에 없는 상품 행만 CBF로 바꾼 것** (제안서 2.1절, 성공 기준 ②의 "fallback 버전")

In [ ]:
# ===== 6. 저장 =====
p_cbf = {'valid': pv[BEST_K], 'test': pt[BEST_K]}
for split in ['valid', 'test']:
    df = getattr(data, split)
    print(' 저장:', M.save_predictions(DS, 'cbf', split, df, p_cbf[split]))
    mf = M.load_predictions(DS, 'mf', split, df)
    un = ~data.seen_mask(df)
    fb = mf.copy(); fb[un] = p_cbf[split][un]
    print(' 저장:', M.save_predictions(DS, 'mf_fb', split, df, fb), f'(CBF로 바꾼 행 {int(un.sum())}개)')
    f_un = f'{PRED_DIR}/{DS}/mf_unseen_{split}.parquet'
    if os.path.exists(f_un):
        n_file = len(pd.read_parquet(f_un))
        print(f'   혜림 님 mf_unseen_{split} 목록 {n_file}행과 {"일치" if n_file == int(un.sum()) else "불일치 → 확인 필요"}')

## 7. Top-N용 유사도 원점수 정의 선택

원점수 = 사용자가 train에서 평가한 상품과의 유사도 중 **상위 score_k개의 평균** (1이면 최댓값).
평가 규칙대로 **valid, 프로토콜 ③(p3_users 1만 명)의 HR@10**으로 고른다. ①②도 같은 1만 명으로 같이 본다.
인기도 기준 모델 값도 같이 출력한다.

먼저 200명으로 시간을 재고 전체 예상 시간을 출력한다.

In [ ]:
# ===== 7-1. 시간 측정 =====
p3 = M.load_p3_users(DS)
t0 = time.time()
_ = cbf.score_rows_ks(p3[:200], SCORE_KS)
per = (time.time() - t0) / 200
print(f'③ 원점수 계산: 사용자 1명당 {per*1000:.1f}ms → {len(p3):,}명 예상 {per*len(p3)/60:.1f}분')

In [ ]:
# ===== 7-2. 원점수 정의 비교 (valid) =====
from src.eval.ranking import target_rank_full
t0 = time.time()
split = 'valid'
target = data.valid.set_index('uidx')['iidx']
rows, hits_score = [], {}

# ①②: 후보 파일에서 p3_users만, 원점수는 정의별로 한 번에 계산
for proto in ['p1', 'p2']:
    cand = M.load_candidates(DS, proto, split)
    cp = cand[cand['uidx'].isin(p3)]
    S = cbf.score_ks(cp['uidx'].values, cp['iidx'].values, SCORE_KS)
    for k in SCORE_KS:
        uu, r = M.rank_candidates(lambda u, i, s=S[k]: s, cand, users=p3)
        hr, nd, h, _ = M.hr_ndcg(r)
        rows.append(dict(score_k=klabel(k), protocol=proto, HR10=hr, NDCG10=nd))

# ③: 사용자 묶음마다 정의 4개를 한 번에 계산 (M.rank_full과 같은 방식·시드)
rngs = {k: np.random.default_rng([SEED, 3]) for k in SCORE_KS}
ranks = {k: np.empty(len(p3), dtype=np.int64) for k in SCORE_KS}
tg = target.reindex(p3).values
CH = 256
for s0 in range(0, len(p3), CH):
    uu = p3[s0:s0 + CH]
    R = cbf.score_rows_ks(uu, SCORE_KS)
    for k in SCORE_KS:
        for r, u in enumerate(uu):
            t = int(tg[s0 + r])
            ranks[k][s0 + r] = target_rank_full(R[k][r], t, data.rated_except(int(u), t), rngs[k])
    if (s0 // CH) % 10 == 0:
        print(f'  ③ {min(s0 + CH, len(p3)):,}/{len(p3):,}명 ({time.time()-t0:.0f}초)')
for k in SCORE_KS:
    hr, nd, h, _ = M.hr_ndcg(ranks[k])
    rows.append(dict(score_k=klabel(k), protocol='p3', HR10=hr, NDCG10=nd))

# 인기도 기준 모델 (같은 사용자)
pop = BASELINES['popularity']().fit(data.train, data.n_users, data.n_items)
tp, _ = M.ranking_eval({'popularity': pop}, data, split, users=p3)
for _, r in tp.iterrows():
    rows.append(dict(score_k='(인기도)', protocol=r['protocol'], HR10=r['HR10'], NDCG10=r['NDCG10']))

score_table = pd.DataFrame(rows)
print(score_table.pivot_table(index='score_k', columns='protocol', values='HR10', sort=False).round(4).to_string())
p3_rows = score_table[(score_table.protocol == 'p3') & (score_table.score_k != '(인기도)')].sort_values(['HR10', 'NDCG10'], ascending=False)
BEST_SCORE_K = SCORE_KS[[klabel(k) for k in SCORE_KS].index(p3_rows.iloc[0]['score_k'])]
cbf.score_k = BEST_SCORE_K
print(f'\n선택: score_k = {klabel(BEST_SCORE_K)}  (valid ③ HR@10 {p3_rows.iloc[0]["HR10"]:.4f})  ({time.time()-t0:.0f}초)')

## 8. 설정·결과 저장

08_hybrid가 이 파일(`results/cbf/{DS}_cbf_config.json`)에서 k와 score_k를 읽는다.

In [ ]:
# ===== 8. 저장 =====
k_table.insert(0, 'dataset', DS); k_table.to_csv(f'{OUT}/{DS}_cbf_k_tuning.csv', index=False)
rt.insert(0, 'dataset', DS); rt.to_csv(f'{OUT}/{DS}_cbf_rating_test.csv', index=False)
score_table.insert(0, 'dataset', DS); score_table.to_csv(f'{OUT}/{DS}_cbf_score_valid.csv', index=False)
cfg = dict(dataset=DS, tfidf=TFIDF, tfidf_path=TFIDF_PATH, tfidf_shape=list(X.shape),
           k=klabel(BEST_K), score_k=klabel(BEST_SCORE_K),
           valid_rmse=float(M.rmse(yv, p_cbf['valid'])), test_rmse=float(M.rmse(yt, p_cbf['test'])),
           test_mae=float(M.mae(yt, p_cbf['test'])))
M.save_json(cfg, f'{OUT}/{DS}_cbf_config.json')
print(json.dumps(cfg, ensure_ascii=False, indent=1))
print(f'\n저장 위치: {OUT}')